# Two-week buffer sensitivity — Experiment 011

Entry rank stays 5; retention ranks are 5 (no buffer), 6, 8 and 10. All variants check membership and restore equal weights only every two weeks; actual quantities remain unchanged between decisions. Both 2022 and 2023 are development data. 2024 and final OOS remain sealed. This notebook reads saved artifacts and never launches a backtest, model fit or download.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'data/interim/buffer_sensitivity/v20261121_001'
prep = json.loads((BASE / 'preparation/summary.json').read_text())
summary_path = BASE / 'results/summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
READY = summary.get('success', False)
assert not prep['sealed_2024_and_final_OOS_inspected']
display(pd.DataFrame(prep['prepared_settings']))
print('Full results available.' if READY else 'Prepared only: run the CLI experiment to compute historical results.')

## Primary comparison

Fees and funding remain identical to the prior two-week baseline. Primary extra slippage is 5bps; 10bps is mandatory stress. Membership changes differ from actual traded notional because retained positions are still resized at decision dates.

In [ ]:
if READY:
    display(pd.read_csv(ROOT / 'reports/tables/buffer_sensitivity/primary_metrics.csv'))
    display(Image(filename=str(ROOT / 'reports/figures/buffer_sensitivity/rank_sensitivity.png')))
else:
    print('No candidate historical performance has been computed yet.')

## Equity, cost stress and concentration

A wider buffer may retain weakening assets. Look for consistency across years and nearby tested ranks, not the single highest return. Best-month exclusion is descriptive, not a tradable timing rule. All 16 cases are retained.

In [ ]:
if READY:
    display(Image(filename=str(ROOT / 'reports/figures/buffer_sensitivity/equity.png')))
    display(Image(filename=str(ROOT / 'reports/figures/buffer_sensitivity/cost_stress.png')))
    display(pd.read_csv(ROOT / 'reports/tables/buffer_sensitivity/monthly_returns.csv'))

## Prespecified gates and verification

A candidate must improve Sharpe, avoid worse MDD, reduce notional and remain profitable outside the best month in both primary years, and remain positive at 10bps. There is no automatic threshold replacement or opening of 2024. Models and scores are unchanged, so original SHAP is reused. Conditional universe, fractional lot, hourly VWAP and funding-mark assumptions still limit interpretation.

In [ ]:
if READY:
    display(pd.read_csv(ROOT / 'reports/tables/buffer_sensitivity/paired_comparisons.csv'))
    display(pd.read_csv(BASE / 'results/all_runs.csv'))
    display(summary)
    display(json.loads((BASE / 'results/verification.json').read_text()))
else:
    print('After completion: python scripts/run_buffer_sensitivity.py --stage summary')